## 파일 준비

In [ ]:
import os
import zipfile
import torch
from torch import nn
from torchvision import transforms
from PIL import Image, ImageOps

checkpoint_path = './best_model.pth'
test_set_dir = './test_set'
test_label_path = './test_labels.txt'

if not os.path.isfile(checkpoint_path) or not os.path.isdir(test_set_dir) or not os.path.isfile(test_label_path):
    from google.colab import files
    files.upload()
    for filename in os.listdir('.'):
        if filename.lower().endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as z:
                z.extractall('.')

if not os.path.isdir(test_set_dir) and os.path.isdir('./train_test_split/test_set'):
    test_set_dir = './train_test_split/test_set'
if not os.path.isfile(test_label_path) and os.path.isfile('./train_test_split/test_labels.txt'):
    test_label_path = './train_test_split/test_labels.txt'

if not os.path.isfile(checkpoint_path):
    raise FileNotFoundError('best_model.pth')
if not os.path.isdir(test_set_dir):
    raise FileNotFoundError('test_set')
if not os.path.isfile(test_label_path):
    raise FileNotFoundError('test_labels.txt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 모델 구조 및 체크포인트 불러오기

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
class_names = checkpoint['class_names']
num_classes = len(class_names)

def create_model(name):
    if name == 'MLP':
        return nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 64, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    if name == 'CNN':
        return nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    raise ValueError(name)

model = create_model(checkpoint['model_name'])
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)
model.eval()

print('Model:', checkpoint['model_name'])
print('Classes:', class_names)

## 이미지 전처리

In [ ]:
def prepare_image(image):
    image = image.convert('L')
    mask = ImageOps.invert(image).point(lambda x: 255 if x > 40 else 0)
    bbox = mask.getbbox()
    if bbox is not None:
        image = image.crop(bbox)
        width, height = image.size
        side = int(max(width, height) * 1.5)
        canvas = Image.new('L', (side, side), 255)
        canvas.paste(image, ((side - width) // 2, (side - height) // 2))
        image = canvas
    return image.resize((64, 64), Image.Resampling.LANCZOS)

test_transform = transforms.Compose([
    transforms.Lambda(prepare_image),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

## 테스트 이미지 및 정답 레이블 불러오기

In [ ]:
gt_labels = {}
with open(test_label_path, 'r', encoding='utf-8-sig') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        filename, label = line.split(':', 1)
        filename = os.path.basename(filename.strip())
        label = label.strip()
        if filename in gt_labels:
            raise ValueError('Duplicate GT filename: ' + filename)
        gt_labels[filename] = label

image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')
test_images = []
for folder, _, filenames in os.walk(test_set_dir):
    for filename in filenames:
        if filename.lower().endswith(image_extensions):
            test_images.append(os.path.join(folder, filename))
test_images.sort()

if not test_images:
    raise ValueError('No test images found')

image_names = [os.path.basename(path) for path in test_images]
if len(image_names) != len(set(image_names)):
    raise ValueError('Duplicate test image filenames')

missing = [name for name in image_names if name not in gt_labels]
if missing:
    raise ValueError('Missing GT labels: ' + str(missing))

unknown = [gt_labels[name] for name in image_names if gt_labels[name] not in class_names]
if unknown:
    raise ValueError('Unknown GT labels: ' + str(unknown))

print('Total test images:', len(test_images))
print('Loaded GT labels:', len(gt_labels))

## 전체 테스트 데이터 정확도 평가

In [ ]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for image_path in test_images:
        filename = os.path.basename(image_path)
        with Image.open(image_path) as image:
            image_tensor = test_transform(image).unsqueeze(0).to(device)

        outputs = model(image_tensor)
        prediction = outputs.argmax(dim=1).item()
        gt = class_names.index(gt_labels[filename])

        correct += int(prediction == gt)
        total += 1

accuracy = correct / total
print('Total:', total)
print('Correct:', correct)
print(f'Accuracy: {accuracy:.4f}')